In [ ]:
import numpy as np
from scipy import stats
import pandas as pd

response_var = "Accuracy"
df = pd.read_csv("../results_pca95_macro.csv")


# Set the confidence level for the calculation of C.I.s
CONFIDENCE_LEVEL = 0.95

#Response array with 3 values for each experimental combination
response = np.array([
    df[(df["PCA"] == -1) & (df["Modelo"] == -1)][response_var].values,
    df[(df["PCA"] ==  1) & (df["Modelo"] == -1)][response_var].values,
    df[(df["PCA"] == -1) & (df["Modelo"] ==  1)][response_var].values,
    df[(df["PCA"] ==  1) & (df["Modelo"] ==  1)][response_var].values
])

# Mean responses for each combination
mean_response = np.mean(response, axis=1)

# Defining A, B, and the interaction AB
A = np.array([-1, 1, -1, 1])
B = np.array([-1, -1, 1, 1])
AB = A * B

replications = response.shape[1]  # Number of replications (3 in this case)


# Calculate the total mean (grand mean) of all responses
overall_mean_response = np.mean(response)
q_0 = overall_mean_response  # Mean effect q_0

# Calculate SSY (Total sum of squares for the responses)
SSY = np.sum(response**2)

# SS0 (Sum of squares for the grand mean)
SS0 = (np.sum(response))**2 / (len(response) * response.shape[1])

# Effects are already calculated earlier and divided by 4 for proper scaling
effect_A = np.sum(A * mean_response) / 4
effect_B = np.sum(B * mean_response) / 4
effect_AB = np.sum(AB * mean_response) / 4

# SSA, SSB, SSAB are based on the squared effects of A, B, and AB, multiplied by the factor scaling (4 * 3 for this case)
SSA = 4 * replications * (effect_A ** 2)
SSB = 4 * replications * (effect_B ** 2)
SSAB = 4 * replications * (effect_AB ** 2)

# SSE (Experimental error sum of squares) is the remaining variation
SSE = SSY - SS0 - SSA - SSB - SSAB

# SST (Total sum of squares) is calculated from SSY - SS0
SST = SSY - SS0

# Variance explained by each factor
var_A = (SSA / SST) * 100
var_B = (SSB / SST) * 100
var_AB = (SSAB / SST) * 100
var_residual = (SSE / SST) * 100

# Standard error of effects
n = 2 ** 2  # Number of factor levels 
se = np.sqrt(SSE / (n * (replications - 1)))

# Variance of the effects
s_qi = se / np.sqrt(n * replications)

# Degrees of freedom for the t-distribution
df = n * (replications - 1)

# Automatically select t or z for the mean effect q_0 based on the number of replications
if replications >= 30:
    # Use z-distribution for large sample sizes
    z_alpha_q0 = stats.norm.ppf(1 - (1 - CONFIDENCE_LEVEL) / 2)
    distribution_type = 'z-distribution'
else:
    # Use t-distribution for smaller sample sizes
    z_alpha_q0 = stats.t.ppf(1 - (1 - CONFIDENCE_LEVEL) / 2, df)
    distribution_type = 't-distribution'

# Standard error for the mean effect q_0 (same formula as effects, but using se directly)
s_q0 = se / np.sqrt(n * replications)

# Calculate the t-critical value for other effects (A, B, AB) for the given confidence interval
t_alpha = stats.t.ppf(1 - (1 - CONFIDENCE_LEVEL) / 2, df)

# Calculate the confidence intervals for each effect
CI_q0 = (round(q_0 - z_alpha_q0 * s_q0,2), q_0, round(q_0 + z_alpha_q0 * s_q0,2))
CI_A = (round(effect_A - t_alpha * s_qi,2), effect_A, round(effect_A + t_alpha * s_qi,2))
CI_B = (round(effect_B - t_alpha * s_qi,2), effect_B, round(effect_B + t_alpha * s_qi,2))
CI_AB = (round(effect_AB - t_alpha * s_qi,2), effect_AB, round(effect_AB + t_alpha * s_qi,2))

# Print results
print(f"SSY: {SSY:.2f}")
print(f"SS0 (Sum of squares of the grand mean): {SS0:.2f}")
print(f"SSA (Sum of squares for A): {SSA:.2f}")
print(f"SSB (Sum of squares for B): {SSB:.2f}")
print(f"SSAB (Sum of squares for interaction A * B): {SSAB:.2f}")
print(f"SSE (Sum of squares for errors): {SSE:.2f}")
print(f"SST (Total sum of squares): {SST:.2f}")

# Print confidence intervals
print(f"\n{CONFIDENCE_LEVEL*100}% Confidence Interval for the mean effect q_0 (using {distribution_type}): {CI_q0}")
print(f"{CONFIDENCE_LEVEL*100}% Confidence Interval for effect A (using {distribution_type}): {CI_A}")
print(f"{CONFIDENCE_LEVEL*100}% Confidence Interval for effect B (using {distribution_type}): {CI_B}")
print(f"{CONFIDENCE_LEVEL*100}% Confidence Interval for interaction A * B (using {distribution_type}): {CI_AB}")

print(f"\nVariance explained by A: {var_A:.2f}%")
print(f"Variance explained by B: {var_B:.2f}%")
print(f"Variance explained by interaction A * B: {var_AB:.2f}%")
print(f"Residual variance (unexplained): {var_residual:.2f}%")



SSY: 13.05
SS0 (Sum of squares of the grand mean): 12.93
SSA (Sum of squares for A): 0.00
SSB (Sum of squares for B): 0.13
SSAB (Sum of squares for interaction A * B): 0.00
SSE (Sum of squares for errors): 0.00
SST (Total sum of squares): 0.13

95.0% Confidence Interval for the mean effect q_0 (using t-distribution): (np.float64(0.8), np.float64(0.8039328699779624), np.float64(0.81))
95.0% Confidence Interval for effect A (using t-distribution): (np.float64(-0.01), np.float64(-0.010357687743685423), np.float64(-0.01))
95.0% Confidence Interval for effect B (using t-distribution): (np.float64(0.08), np.float64(0.07931852856416349), np.float64(0.08))
95.0% Confidence Interval for interaction A * B (using t-distribution): (np.float64(-0.0), np.float64(-0.001084929649093097), np.float64(0.0))

Variance explained by A: 1.67%
Variance explained by B: 98.19%
Variance explained by interaction A * B: 0.02%
Residual variance (unexplained): 0.12%
